# COS30082 Assignment 1: Bird Species Classification (CUB-200)

**Name:** Zwe (edit to your full name) &nbsp;|&nbsp; **Student ID:** 102783659

## Goal
Classify 200 bird species with only ~24 training images per class. This is a *fine-grained* problem:
many species differ only in small details (beak shape, wing bars, eye rings), so the model has to see detail
and must not overfit the tiny dataset.

## Experiment: one backbone, two training configurations
Both runs use a **ConvNeXt pretrained on ImageNet-1k**, the same data split, the same seed and the same
optimiser, so any difference in results comes from the configuration. The two configurations are run for
each model (set `MODEL` in section 4), so model size and pretraining data can be compared as well:
ConvNeXt-Tiny pretrained on ImageNet-1k (torchvision) and ConvNeXt-Base pretrained on ImageNet-22k (timm).

| Setting | Config A: Standard fine-tune | Config B: Fine-grained fine-tune |
|---|---|---|
| Input resolution | 224 px throughout | Progressive: 224 px, then **448 px** |
| Epochs | 20 | 15 at 224 px + 5 at 448 px |
| Batch size | 32 | 32 at 224 px, 16 at 448 px |
| Train augmentation | RandomResizedCrop, flip, rotation, colour jitter, random erasing | Same **+ CutMix** |
| Eval preprocessing | Resize 256 → CenterCrop 224 | Resize 512 → CenterCrop 448 |
| Test-time augmentation | Reported with and without | Reported with and without |
| Loss | Cross-entropy, label smoothing 0.1 | Same (applied to CutMix's mixed labels) |
| Optimiser | AdamW, weight decay 0.05 | AdamW, weight decay 0.05 |
| Learning rate | backbone 1e-4, head 1e-3 | Same, then backbone 2e-5 / head 2e-4 at 448 px |
| LR schedule | 1 warm-up epoch, then cosine | Same, plus a fresh cosine cycle at 448 px |
| Regularisation | Dropout 0.3 (head), stochastic depth 0.1 | Same + CutMix |
| Early stopping | Patience 6 on validation accuracy | Patience 6 on validation accuracy |

An optional ablation (`B_no_cutmix`) separates the effect of resolution from the effect of CutMix.

**Rule followed:** the validation set is used for every decision (early stopping, checkpoint and model
selection, TTA on/off). The test set is only used for the final reported numbers.

**Metrics:** Top-1 accuracy and average accuracy per class (both required by the brief).

## 1. Setup

In [1]:
import os, re, json, time, math, random, zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import timm                 # PyTorch Image Models: provides ConvNeXt weights pretrained on ImageNet-22k

# Reproducibility: fix every random number generator we use
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"          # mixed precision (float16) = faster training, less GPU memory
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("torch", torch.__version__)

Device: cuda NVIDIA GeForce RTX 4060 Laptop GPU
torch 2.14.1+cu130


/mnt/hyhyData/Swinburne/Applied Machine Learning/Assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data
Everything lives in the assignment folder on this machine: `train.txt` / `test.txt`, the unzipped
`Train/` and `Test/` image folders, and the outputs (`checkpoints/`, `hf_space/`).

In [2]:
# ---- Paths: the assignment folder on this machine ----
DATA_DIR   = "/mnt/hyhyData/Swinburne/Applied Machine Learning/Assignment"  # train.txt, test.txt, zips
WORK_DIR   = DATA_DIR                                 # Train/ and Test/ image folders are here
CKPT_DIR   = os.path.join(DATA_DIR, "checkpoints")    # model weights + training histories
DEPLOY_DIR = os.path.join(DATA_DIR, "hf_space")       # files to upload to Hugging Face
for d in (CKPT_DIR, DEPLOY_DIR):
    os.makedirs(d, exist_ok=True)

for f in ["train.txt", "test.txt"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} not found in {DATA_DIR}"

# Unzip only if the image folders are missing (they are already extracted here)
for z in ["Train.zip", "Test.zip"]:
    target = os.path.join(WORK_DIR, z.replace(".zip", ""))
    if not os.path.isdir(target):
        with zipfile.ZipFile(os.path.join(DATA_DIR, z)) as f:
            f.extractall(target)
        print("Extracted", z)

print(f"DATA_DIR = {DATA_DIR}\nWORK_DIR = {WORK_DIR}")

DATA_DIR = /mnt/hyhyData/Swinburne/Applied Machine Learning/Assignment
WORK_DIR = /mnt/hyhyData/Swinburne/Applied Machine Learning/Assignment


In [3]:
def read_annotations(path):
    """Each line is: <image name> <class label>. Returns a DataFrame with columns file, label."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                name, label = line.rsplit(" ", 1)   # split on the LAST space only
                rows.append((name, int(label)))
    return pd.DataFrame(rows, columns=["file", "label"])

def index_images(root):
    """Map file name -> full path, so it works whatever folder layout the zip has."""
    idx = {}
    for d, _, files in os.walk(root):
        if "__MACOSX" in d:                     # skip macOS zip metadata
            continue
        for fn in files:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                idx[fn] = os.path.join(d, fn)
    return idx

full_df = read_annotations(os.path.join(DATA_DIR, "train.txt"))
test_df = read_annotations(os.path.join(DATA_DIR, "test.txt"))

train_idx = index_images(os.path.join(WORK_DIR, "Train"))
test_idx  = index_images(os.path.join(WORK_DIR, "Test"))
full_df["path"] = full_df.file.map(train_idx)
test_df["path"] = test_df.file.map(test_idx)
assert full_df.path.notna().all(), "Some training images were not found"
assert test_df.path.notna().all(), "Some test images were not found"

NUM_CLASSES = int(full_df.label.max()) + 1

# Species names come from the training file names, e.g.
#   Black_footed_Albatross_0019_416160254.jpg -> "Black footed Albatross"
# 91 files end in "_xxx.jpg" instead of a number, so the last token is matched as [^_]+ (anything).
def species_name(fname):
    return re.sub(r"_\d+_[^_]+\.jpg$", "", fname).replace("_", " ")

# Safety check: every label must map to exactly one species name
names_per_label = full_df.groupby("label").file.apply(lambda s: sorted(set(map(species_name, s))))
bad = names_per_label[names_per_label.str.len() != 1]
assert bad.empty, f"Labels with inconsistent names:\n{bad}"
class_names = [names_per_label[i][0] for i in range(NUM_CLASSES)]

counts = full_df.label.value_counts()
print(f"Train images: {len(full_df)} | Test images: {len(test_df)} | Classes: {NUM_CLASSES}")
print(f"Images per class (train): min {counts.min()}, max {counts.max()}, mean {counts.mean():.1f}")

# Stratified 80/20 split so every class appears in both train and validation
train_df, val_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df.label, random_state=SEED)
print(f"Split -> train: {len(train_df)}, val: {len(val_df)}")

Train images: 4829 | Test images: 1204 | Classes: 200
Images per class (train): min 16, max 31, mean 24.1
Split -> train: 3863, val: 966


## 3. Transforms and DataLoaders
Transforms are built for a given image size, because Config B switches from 224 px to 448 px.
Augmentation is applied to the **training set only**. Validation, test and the web app use the plain
resize + centre-crop (resize to `size * 256/224`, i.e. 256 for 224 and 512 for 448), with the ImageNet
mean/std that the pretrained weights expect.

In [4]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
NUM_WORKERS = 2

def make_transforms(img_size):
    """Returns (train transform, eval transform) for a square input of img_size pixels."""
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),  # random zoom / crop
        transforms.RandomHorizontalFlip(),                          # birds face either way
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25),                           # hides a random patch
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(img_size * 256 / 224)),               # keep aspect ratio
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, eval_tf

class BirdDataset(Dataset):
    """Loads an image from disk and applies a transform. Returns (image tensor, label)."""
    def __init__(self, df, tf):
        self.paths = df.path.tolist()
        self.labels = df.label.tolist()
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")   # some JPEGs are greyscale
        return self.tf(img), self.labels[i]

def make_loader(df, img_size, batch_size, train):
    """Training loaders shuffle and augment; evaluation loaders do neither."""
    train_tf, eval_tf = make_transforms(img_size)
    return DataLoader(BirdDataset(df, train_tf if train else eval_tf),
                      batch_size=batch_size, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))

## 4. The two configurations
All settings live here, so the report table can be checked against the code.
`RETRAIN = False` means a run that already has a checkpoint and history in `checkpoints/` is loaded
instead of being trained again (useful if the kernel restarts).

In [5]:
# ---- Which model to train in this session ----
# Names with a "." are timm models ("<architecture>.<pretrained weights>"); names without one are
# torchvision models with ImageNet-1k weights. Runs are saved as "<MODEL>_<config>", so results from
# different models are kept side by side.
#   "convnext_tiny"             torchvision, ImageNet-1k  (1.28 M images, 1,000 classes), 28 M params
#   "convnext_base.fb_in22k"    timm, ImageNet-22k (~14 M images, 21,841 classes),       88 M params
MODEL = "convnext_base.fb_in22k"
ARCH = MODEL.split(".")[0]  # architecture only, e.g. "convnext_base" (used for the batch-size table)

# Largest batch that fits in GPU memory per variant and resolution (measured on an 8 GB RTX 4060 Laptop
# GPU with mixed precision). If a config asks for a bigger batch, gradients are accumulated over several
# smaller batches, so the *effective* batch size (and the results) stay the same.
# ConvNeXt uses LayerNorm, not BatchNorm, so accumulation is equivalent to one large batch.
MAX_BATCH = {"convnext_tiny":  {224: 32, 448: 16},
             "convnext_small": {224: 32, 448: 8},
             "convnext_base":  {224: 32, 448: 8}}

# Shared by every run (only the settings in CONFIGS differ)
COMMON = dict(
    label_smoothing=0.1,   # softens the one-hot targets, discourages over-confident predictions
    weight_decay=0.05,     # AdamW decoupled weight decay (standard value for ConvNeXt fine-tuning)
    dropout=0.3,           # dropout before the new 200-class layer
    drop_path=0.1,         # stochastic depth: randomly skips residual blocks during training
    patience=6,            # early stopping: epochs without val improvement before stopping a phase
)

# A "phase" = a block of epochs at one resolution with its own LR schedule.
CONFIGS = {
    "A_standard": dict(
        cutmix_prob=0.0,
        phases=[dict(img_size=224, batch_size=32, epochs=20,
                     lr_backbone=1e-4, lr_head=1e-3, warmup_epochs=1)],
    ),
    "B_finegrained": dict(
        cutmix_prob=0.5,   # CutMix is applied to half of the training batches
        phases=[dict(img_size=224, batch_size=32, epochs=15,
                     lr_backbone=1e-4, lr_head=1e-3, warmup_epochs=1),
                # Progressive resizing: continue from the best 224 px weights at 448 px,
                # with a lower LR because the model is already well trained
                dict(img_size=448, batch_size=16, epochs=5,
                     lr_backbone=2e-5, lr_head=2e-4, warmup_epochs=0)],
    ),
}

# Optional ablation: Config B without CutMix (isolates the effect of the higher resolution)
RUN_ABLATION = False
ABLATION = {"B_no_cutmix": {**CONFIGS["B_finegrained"], "cutmix_prob": 0.0}}

RETRAIN = False
run_name  = lambda cfg_name, model=MODEL: f"{model}_{cfg_name}"   # e.g. "convnext_base.fb_in22k_A_standard"
ckpt_path = lambda name: os.path.join(CKPT_DIR, f"{name}.pt")
hist_path = lambda name: os.path.join(CKPT_DIR, f"{name}_history.json")
print("Training model:", MODEL)

Training model: convnext_base.fb_in22k


## 5. Model: ConvNeXt
ConvNeXt is a modern CNN (Liu et al., 2022) that borrows design choices from Vision Transformers
(large 7×7 depthwise kernels, LayerNorm, GELU, inverted bottlenecks). Being a CNN, it accepts 448 px
input without any change to the architecture.

| Model | Blocks per stage | Channels | Params | Pretraining data | Library |
|---|---|---|---|---|---|
| ConvNeXt-Tiny | 3-3-9-3 | 96 → 768 | 28 M | ImageNet-1k: 1.28 M images, 1,000 classes | torchvision |
| ConvNeXt-Base | 3-3-27-3 | 128 → 1024 | 88 M | ImageNet-22k: ~14 M images, 21,841 classes | timm (`fb_in22k`) |

ImageNet-22k is about 11× larger than ImageNet-1k and contains many more fine-grained categories
(including hundreds of bird species), so its features should transfer better to CUB-200.

We keep the pretrained feature extractor and replace only the final classification layer with a new
`Dropout(0.3) → Linear(features, 200)` head (768 features for Tiny, 1024 for Base).

In [6]:
def build_convnext(model_name=MODEL, num_classes=NUM_CLASSES, pretrained=True,
                   dropout=COMMON["dropout"], drop_path=COMMON["drop_path"]):
    if "." in model_name:
        # timm: num_classes replaces the 21,841-class ImageNet-22k layer with a new 200-class layer,
        # drop_rate puts dropout just before it, drop_path_rate is stochastic depth
        return timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes,
                                 drop_rate=dropout, drop_path_rate=drop_path)
    # torchvision (ImageNet-1k), e.g. models.convnext_tiny(weights="IMAGENET1K_V1")
    m = getattr(models, model_name)(weights="IMAGENET1K_V1" if pretrained else None,
                                    stochastic_depth_prob=drop_path)
    # classifier = [LayerNorm2d, Flatten, Linear(features -> 1000)]; replace the last layer
    in_f = m.classifier[2].in_features
    m.classifier[2] = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

def head_layer(m):
    """The new, randomly initialised 200-class layer (timm: head.fc, torchvision: classifier[2])."""
    return m.get_classifier() if hasattr(m, "get_classifier") else m.classifier[2]

def split_params(m):
    """Returns (backbone params, head params) so they can get different learning rates."""
    head = list(head_layer(m).parameters())
    head_ids = {id(p) for p in head}
    backbone = [p for p in m.parameters() if id(p) not in head_ids]
    return backbone, head

_m = build_convnext()
print(f"{MODEL} parameters: {sum(p.numel() for p in _m.parameters())/1e6:.1f} M")
print("Output shape check:", _m.eval()(torch.randn(1, 3, 224, 224)).shape)
del _m

convnext_base.fb_in22k parameters: 87.8 M
Output shape check: torch.Size([1, 200])


## 6. Helpers: CutMix, LR schedule, evaluation (with optional TTA), metrics
* **CutMix** (Yun et al., 2019): a random box from another image in the batch is pasted in, and the
  loss is mixed in proportion to the box area. The model cannot rely on one small region, which reduces
  overfitting.
* **Warm-up + cosine LR**: the LR rises linearly during warm-up (so the random head does not damage the
  pretrained weights), then decays smoothly to zero. Updated every batch.
* **TTA** (test-time augmentation): average the predicted probabilities of the image and its mirror image.

In [7]:
def cutmix(imgs, labels, alpha=1.0):
    """Returns (mixed images, original labels, pasted labels, lam = fraction of the original kept)."""
    lam = np.random.beta(alpha, alpha)                    # target mixing ratio
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    H, W = imgs.shape[2], imgs.shape[3]
    cut_h, cut_w = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)   # box centre
    y1, y2 = int(np.clip(cy - cut_h // 2, 0, H)), int(np.clip(cy + cut_h // 2, 0, H))
    x1, x2 = int(np.clip(cx - cut_w // 2, 0, W)), int(np.clip(cx + cut_w // 2, 0, W))
    imgs = imgs.clone()
    imgs[:, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)             # exact ratio after clipping to the image
    return imgs, labels, labels[perm], lam

def warmup_cosine(optimizer, total_steps, warmup_steps):
    """LR multiplier: linear warm-up, then cosine decay to 0. Stepped once per batch."""
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)

@torch.no_grad()
def predict_probs(model, loader, tta=False):
    """Class probabilities for every image in the loader. With tta=True, averages image + mirror."""
    model.eval()                         # disables dropout / stochastic depth
    all_p, all_y = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type=device, enabled=use_amp):
            p = F.softmax(model(imgs).float(), dim=1)
            if tta:
                p_flip = F.softmax(model(torch.flip(imgs, dims=[3])).float(), dim=1)
                p = (p + p_flip) / 2
        all_p.append(p.cpu()); all_y.append(labels)
    return torch.cat(all_p), torch.cat(all_y)

def evaluate(model, loader, tta=False):
    """Returns (cross-entropy loss, top-1 accuracy). Loss uses plain CE (no label smoothing)."""
    probs, y = predict_probs(model, loader, tta)
    loss = F.nll_loss(torch.log(probs.clamp_min(1e-8)), y).item()
    acc = (probs.argmax(1) == y).float().mean().item()
    return loss, acc

def compute_metrics(y_true, y_pred, num_classes=NUM_CLASSES):
    """The two required metrics: Top-1 accuracy and average accuracy per class."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    top1 = (y_true == y_pred).mean()
    per_class = np.full(num_classes, np.nan)
    for c in range(num_classes):
        mask = y_true == c
        if mask.any():
            per_class[c] = (y_pred[mask] == c).mean()     # T_i in the brief
    return float(top1), float(np.nanmean(per_class)), per_class

## 7. Training loop
One function trains any configuration, so A and B are treated identically apart from their settings.
For each phase it builds new loaders (at that phase's resolution), a new optimiser and a new LR schedule.
The best checkpoint (by validation accuracy) is saved to `checkpoints/` with the resolution it was evaluated at,
and the next phase starts from the best weights so far.

In [8]:
def train_model(name, cfg, model_name=MODEL):
    set_seed()
    arch = model_name.split(".")[0]           # for the batch-size table
    model = build_convnext(model_name).to(device)
    backbone, head = split_params(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=COMMON["label_smoothing"])

    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [],
            "img_size": [], "phase_starts": []}
    best = {"val_acc": -1.0}
    t0 = time.time()

    for p_i, ph in enumerate(cfg["phases"], 1):
        hist["phase_starts"].append(len(hist["val_acc"]))   # for the vertical lines in the plots
        # Gradient accumulation: if the config's batch does not fit in GPU memory, use smaller
        # "micro-batches" and only update the weights after `accum` of them (same effective batch).
        micro_bs = min(ph["batch_size"], MAX_BATCH[arch][ph["img_size"]])
        accum = ph["batch_size"] // micro_bs
        assert micro_bs * accum == ph["batch_size"], "batch_size must be a multiple of the micro-batch"
        train_loader = make_loader(train_df, ph["img_size"], micro_bs, train=True)
        val_loader   = make_loader(val_df,   ph["img_size"], micro_bs, train=False)
        steps_per_epoch = len(train_loader) // accum            # optimiser updates per epoch
        print(f"[{name}] phase {p_i}: {ph['img_size']}px, batch {ph['batch_size']} "
              f"(= {accum} x {micro_bs}), {steps_per_epoch} updates/epoch")

        # Two parameter groups: small LR for pretrained layers, larger LR for the new head
        optimizer = torch.optim.AdamW(
            [{"params": backbone, "lr": ph["lr_backbone"]},
             {"params": head,     "lr": ph["lr_head"]}],
            weight_decay=COMMON["weight_decay"])
        scheduler = warmup_cosine(optimizer, ph["epochs"] * steps_per_epoch,
                                  ph["warmup_epochs"] * steps_per_epoch)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)   # avoids float16 underflow
        bad_epochs = 0

        for epoch in range(1, ph["epochs"] + 1):
            model.train()                         # enables dropout and stochastic depth
            run_loss, run_correct, seen = 0.0, 0.0, 0
            optimizer.zero_grad(set_to_none=True)
            for i, (imgs, labels) in enumerate(train_loader):
                if i >= steps_per_epoch * accum:  # drop an incomplete accumulation group
                    break
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                use_cm = np.random.rand() < cfg["cutmix_prob"]
                if use_cm:
                    imgs, y_a, y_b, lam = cutmix(imgs, labels)

                with torch.autocast(device_type=device, enabled=use_amp):
                    logits = model(imgs)
                    if use_cm:   # loss weighted by how much of each image is visible
                        loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
                    else:
                        loss = criterion(logits, labels)
                # backpropagation; dividing by accum makes the summed gradients an average
                scaler.scale(loss / accum).backward()
                if (i + 1) % accum == 0:          # every `accum` micro-batches:
                    scaler.step(optimizer)        #   weight update
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()              #   LR schedule is per update

                preds = logits.argmax(1)
                if use_cm:   # "soft" accuracy on mixed images (only used for the curves)
                    correct = lam * (preds == y_a).sum().item() + (1 - lam) * (preds == y_b).sum().item()
                else:
                    correct = (preds == labels).sum().item()
                run_loss += loss.item() * labels.size(0)
                run_correct += correct
                seen += labels.size(0)

            val_loss, val_acc = evaluate(model, val_loader)
            hist["train_loss"].append(run_loss / seen); hist["train_acc"].append(run_correct / seen)
            hist["val_loss"].append(val_loss);          hist["val_acc"].append(val_acc)
            hist["img_size"].append(ph["img_size"])
            print(f"[{name}] phase {p_i} ({ph['img_size']}px) epoch {epoch:02d}/{ph['epochs']} | "
                  f"train loss {hist['train_loss'][-1]:.3f} acc {hist['train_acc'][-1]:.3f} | "
                  f"val loss {val_loss:.3f} acc {val_acc:.3f}")

            if val_acc > best["val_acc"]:         # new best: save it to disk
                best = {"arch": model_name, "val_acc": val_acc, "img_size": ph["img_size"],
                        "epoch": len(hist["val_acc"]) - 1,
                        "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
                torch.save(best, ckpt_path(name))
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= COMMON["patience"]:
                    print(f"[{name}] early stopping in phase {p_i}")
                    break

        model.load_state_dict(best["state_dict"])  # next phase starts from the best weights

    hist.update(arch=model_name, best_epoch=best["epoch"], best_val_acc=best["val_acc"],
                best_img_size=best["img_size"], minutes=(time.time() - t0) / 60)
    with open(hist_path(name), "w") as f:
        json.dump(hist, f)
    print(f"[{name}] best val acc {best['val_acc']:.4f} at epoch {best['epoch']+1} "
          f"({best['img_size']}px) | {hist['minutes']:.1f} min")
    del model; torch.cuda.empty_cache()
    return hist

def plot_history(hist, title):
    """Loss and accuracy curves; dashed lines mark the start of each new phase (resolution change)."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for a, key in zip(ax, ["loss", "acc"]):
        a.plot(hist[f"train_{key}"], label="train"); a.plot(hist[f"val_{key}"], label="val")
        for s in hist["phase_starts"][1:]:
            a.axvline(s - 0.5, ls="--", c="grey", lw=1)
        a.set_title(f"{title}: {key}"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def run_or_load(name, cfg):
    """Train the run, or load it from checkpoints/ if it already finished (unless RETRAIN is True)."""
    if not RETRAIN and os.path.exists(hist_path(name)) and os.path.exists(ckpt_path(name)):
        print(f"[{name}] found a finished run in checkpoints/, loading it (set RETRAIN = True to retrain)")
        with open(hist_path(name)) as f:
            return json.load(f)
    return train_model(name, cfg)

histories = {}

## 8. Train Config A: standard fine-tune (224 px)

In [9]:
name = run_name("A_standard")
histories[name] = run_or_load(name, CONFIGS["A_standard"])
plot_history(histories[name], f"{MODEL}: Config A")

[convnext_base.fb_in22k_A_standard] phase 1: 224px, batch 32 (= 1 x 32), 120 updates/epoch


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 01/20 | train loss 4.216 acc 0.267 | val loss 1.297 acc 0.719


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 02/20 | train loss 1.571 acc 0.799 | val loss 0.497 acc 0.889


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 03/20 | train loss 1.212 acc 0.921 | val loss 0.440 acc 0.896


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 04/20 | train loss 1.120 acc 0.948 | val loss 0.394 acc 0.914


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 05/20 | train loss 1.073 acc 0.965 | val loss 0.404 acc 0.909


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 06/20 | train loss 1.049 acc 0.968 | val loss 0.414 acc 0.921


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 07/20 | train loss 1.023 acc 0.977 | val loss 0.403 acc 0.925


[convnext_base.fb_in22k_A_standard] phase 1 (224px) epoch 08/20 | train loss 1.011 acc 0.981 | val loss 0.398 acc 0.925


## 9. Train Config B: fine-grained fine-tune (224 → 448 px, CutMix)

In [ ]:
name = run_name("B_finegrained")
histories[name] = run_or_load(name, CONFIGS["B_finegrained"])
plot_history(histories[name], f"{MODEL}: Config B")

### Optional ablation: Config B without CutMix
Set `RUN_ABLATION = True` in section 4 if time allows. Comparing it with A and B shows how much of
the gain comes from resolution and how much from CutMix.

In [ ]:
if RUN_ABLATION:
    for cfg_name, cfg in ABLATION.items():
        name = run_name(cfg_name)
        histories[name] = run_or_load(name, cfg)
        plot_history(histories[name], name)
else:
    print("Ablation skipped (RUN_ABLATION = False)")

## 10. Results
Every finished run found in `checkpoints/` is evaluated (all ConvNeXt variants trained so far, not only
the one in `MODEL`), so variants can be compared in one table. Each run's best checkpoint is evaluated
**with and without TTA** (TTA needs no retraining).

* `train_acc`: accuracy on the training images with *eval* preprocessing (no augmentation, no CutMix),
  so it is comparable between A and B.
* `gap`: `train_acc - val_top1`, a measure of overfitting.
* `test_top1` and `test_avg_per_class`: the two metrics required by the brief.

In [ ]:
def load_best(name):
    """Rebuilds the right architecture and loads the best checkpoint saved during training."""
    ckpt = torch.load(ckpt_path(name), map_location="cpu")
    # Older checkpoints did not store "arch"; it is the run name minus the config suffix
    ckpt.setdefault("arch", next(name[:-len(c) - 1] for c in ALL_CONFIGS if name.endswith("_" + c)))
    m = build_convnext(ckpt["arch"], pretrained=False)
    m.load_state_dict(ckpt["state_dict"])
    return m.to(device), ckpt

# Collect every finished run on disk ("<model>_<config>_history.json" + "<model>_<config>.pt")
ALL_CONFIGS = list(CONFIGS) + list(ABLATION)
for fn in sorted(os.listdir(CKPT_DIR)):
    if fn.endswith("_history.json"):
        name = fn[:-len("_history.json")]
        if name not in histories and os.path.exists(ckpt_path(name)):
            with open(hist_path(name)) as f:
                histories[name] = json.load(f)
print("Runs evaluated:", list(histories))

EVAL_BS = 16
rows, test_outputs = [], {}

for name, h in histories.items():
    model, ckpt = load_best(name)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    size = ckpt["img_size"]                          # evaluate at the resolution of the best epoch
    loaders = {split: make_loader(df, size, EVAL_BS, train=False)
               for split, df in [("train", train_df), ("val", val_df), ("test", test_df)]}
    _, train_acc = evaluate(model, loaders["train"])

    for tta in (False, True):
        label = name + (" + TTA" if tta else "")
        _, val_acc = evaluate(model, loaders["val"], tta)
        probs, y = predict_probs(model, loaders["test"], tta)
        yt, yp = y.numpy(), probs.argmax(1).numpy()
        top1, avg_cls, per_class = compute_metrics(yt, yp)
        test_outputs[label] = (yt, yp, per_class)
        rows.append({"run": label, "params_M": round(n_params, 1),
                     "eval_size": size, "best_epoch": ckpt["epoch"] + 1,
                     "train_acc": train_acc, "val_top1": val_acc, "gap": train_acc - val_acc,
                     "test_top1": top1, "test_avg_per_class": avg_cls,
                     "train_minutes": h["minutes"]})
    del model; torch.cuda.empty_cache()

results_df = pd.DataFrame(rows).set_index("run").round(4)
display(results_df)
results_df.to_csv(os.path.join(CKPT_DIR, "results.csv"))

In [ ]:
# Validation accuracy per epoch for every run (good figure for the report)
plt.figure(figsize=(8, 4.5))
for name, h in histories.items():
    plt.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Validation accuracy per epoch"); plt.show()

## 11. Error analysis (best run)
The best run is chosen by **validation** accuracy, never by test accuracy.

In [ ]:
best_label = results_df.val_top1.idxmax()
best_run = best_label.replace(" + TTA", "")
best_tta = best_label.endswith("+ TTA")
print(f"Best by validation accuracy: {best_label}")
yt, yp, per_class = test_outputs[best_label]

# Distribution of per-class accuracy: shows whether errors are spread out or concentrated
plt.figure(figsize=(6, 3))
plt.hist(per_class, bins=20)
plt.xlabel("test accuracy of a class"); plt.ylabel("number of classes")
plt.title(f"Per-class accuracy ({best_label})"); plt.show()

worst = np.argsort(per_class)[:10]
print("\nWorst 10 classes:")
for c in worst:
    print(f"  {per_class[c]:.2f}  {class_names[c]}")

# Most common confusions (true -> predicted)
pairs = Counter((t, p) for t, p in zip(yt, yp) if t != p)
print("\nMost common confusions:")
for (t, p), n in pairs.most_common(10):
    print(f"  {n}x  {class_names[t]}  ->  {class_names[p]}")

In [ ]:
# Show one training example of each species in the top-4 confusions (why the model mixes them up)
top_pairs = [tp for tp, _ in pairs.most_common(4)]
if top_pairs:
    fig, ax = plt.subplots(len(top_pairs), 2, figsize=(6, 3 * len(top_pairs)), squeeze=False)
    for r, (t, p) in enumerate(top_pairs):
        for c, (cls, tag) in enumerate([(t, "true"), (p, "predicted")]):
            path = full_df[full_df.label == cls].path.iloc[0]
            ax[r, c].imshow(Image.open(path).convert("RGB"))
            ax[r, c].set_title(f"{tag}: {class_names[cls]}", fontsize=9); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

## 12. Save the best model and write the Hugging Face files
`model.pt` stores the weights, class names, input size and whether TTA is used, so `app.py` reproduces
exactly the evaluation setup above. Everything is written to the `hf_space/` folder.

In [ ]:
ckpt = torch.load(ckpt_path(best_run), map_location="cpu")
torch.save({"arch": ckpt["arch"], "state_dict": ckpt["state_dict"], "class_names": class_names,
            "img_size": ckpt["img_size"], "tta": best_tta},
           os.path.join(DEPLOY_DIR, "model.pt"))
print(f"Saved {best_label} ({ckpt['img_size']}px, TTA={best_tta}) to", os.path.join(DEPLOY_DIR, "model.pt"))

In [ ]:
APP_CODE = r"""
# Gradio app for Hugging Face Spaces: upload a bird photo, get the top-5 predicted species.
import torch, torch.nn as nn, torch.nn.functional as F, gradio as gr, timm
from torchvision import models, transforms

ckpt = torch.load("model.pt", map_location="cpu")
class_names = ckpt["class_names"]
img_size = ckpt["img_size"]          # 224 or 448, whichever the best checkpoint used
use_tta = ckpt["tta"]                # average image + mirror image, as in the notebook

# Same architecture as the notebook; no pretrained download, the weights come from model.pt
arch = ckpt["arch"]
if "." in arch:      # timm model, e.g. "convnext_base.fb_in22k"
    model = timm.create_model(arch, pretrained=False, num_classes=len(class_names))
else:                # torchvision model, e.g. "convnext_tiny"
    model = getattr(models, arch)(weights=None)
    model.classifier[2] = nn.Sequential(nn.Dropout(0.3),
                                        nn.Linear(model.classifier[2].in_features, len(class_names)))
model.load_state_dict(ckpt["state_dict"])
model.eval()

# Must be identical to the evaluation transform in the notebook
tf = transforms.Compose([
    transforms.Resize(int(img_size * 256 / 224)), transforms.CenterCrop(img_size),
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(img):
    if img is None:
        return {}
    x = tf(img.convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
        if use_tta:
            probs = (probs + F.softmax(model(torch.flip(x, dims=[3])), dim=1)) / 2
    probs = probs[0]
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload a bird photo"),
    outputs=gr.Label(num_top_classes=5, label="Predicted species"),
    title="CUB-200 Bird Species Classifier",
    description=f"COS30082 Assignment 1 ({ckpt['arch']}). Upload a bird photo to see the top-5 predicted species.",
)
demo.launch()
"""
# Written with Python (not %%writefile) because the folder path contains spaces
with open(os.path.join(DEPLOY_DIR, "app.py"), "w") as f:
    f.write(APP_CODE.lstrip())
print("Wrote", os.path.join(DEPLOY_DIR, "app.py"))

In [ ]:
with open(os.path.join(DEPLOY_DIR, "requirements.txt"), "w") as f:
    f.write("torch\ntorchvision\ntimm\ngradio\npillow\n")
print("Wrote", os.path.join(DEPLOY_DIR, "requirements.txt"))

## 13. Deploying to Hugging Face Spaces
1. Create a new **Space** (SDK: Gradio, hardware: CPU basic is enough).
2. Upload `app.py`, `requirements.txt` and `model.pt` from the `hf_space/` folder. `model.pt` is
   about 110 MB for Tiny and 350 MB for Base, so upload it through the web UI ("Add file" → "Upload files"), which handles large files.
3. Wait for the build, then test with a few photos from the test set.
4. Put the Space link and your YouTube link in the report.